In [1]:
from google.colab import files
files.upload()

Saving drc_zone_table_v5.csv to drc_zone_table_v5.csv


{'drc_zone_table_v5.csv': b'province,health_zone,pcode,population_2022,confirmed_cases,recorded_deaths,first_confirmed,affected,cases_per_100k,log10_cases_per_100k_plus1,days_until_first_case,rwi_mean,rwi_points,rwi_error_mean,centroid_lon,centroid_lat,area_km2,km_to_mongbwalu,pop_density_per_km2,travel_time_min,ndvi_crop_recent,ndvi_crop_normal,ndvi_crop_anomaly,ndvi_all_recent,ndvi_all_normal,ndvi_all_anomaly,ndvi_crop_missing,ndvi_anomaly_filled,stunting_2013_pct,stunting_pixels\nIturi,Bunia,CD5402ZS02,301886,2019,707,2026-05-18,1,668.8,2.826,3.0,-0.042,60,0.425,30.2234,1.5955,369.0,55.1,817.9,7.4,0.5995,0.6088,-0.0093,0.6265,0.6307,-0.0042,0,-0.0093,52.4,18\nIturi,Rwampara,CD5402ZS06,150948,1206,436,2026-05-15,1,798.95,2.903,0.0,-0.333,62,0.382,30.1102,1.5719,470.0,50.6,321.0,33.4,0.6679,0.6708,-0.0029,0.7003,0.6978,0.0024,0,-0.0029,57.1,24\nIturi,Nizi,CD5405ZS11,159485,855,313,2026-05-21,1,536.1,2.73,6.0,-0.231,39,0.397,30.3247,1.7497,240.0,52.4,664.3,7.7,0.69,0.6756,0.0144,0.7097

In [2]:
import pandas as pd
df = pd.read_csv('drc_zone_table_v5.csv')
df.head()

,province,health_zone,pcode,population_2022,confirmed_cases,recorded_deaths,first_confirmed,affected,cases_per_100k,log10_cases_per_100k_plus1,...,ndvi_crop_recent,ndvi_crop_normal,ndvi_crop_anomaly,ndvi_all_recent,ndvi_all_normal,ndvi_all_anomaly,ndvi_crop_missing,ndvi_anomaly_filled,stunting_2013_pct,stunting_pixels
0,Ituri,Bunia,CD5402ZS02,301886,2019,707,2026-05-18,1,668.80,2.826,...,0.5995,0.6088,-0.0093,0.6265,0.6307,-0.0042,0,-0.0093,52.4,18
1,Ituri,Rwampara,CD5402ZS06,150948,1206,436,2026-05-15,1,798.95,2.903,...,0.6679,0.6708,-0.0029,0.7003,0.6978,0.0024,0,-0.0029,57.1,24
2,Ituri,Nizi,CD5405ZS11,159485,855,313,2026-05-21,1,536.10,2.730,...,0.6900,0.6756,0.0144,0.7097,0.6936,0.0160,0,0.0144,54.7,12
3,Nord-Kivu,Beni,CD6109ZS01,449812,737,485,2026-06-01,1,163.85,2.217,...,0.6289,0.6347,-0.0058,0.4077,0.4393,-0.0316,0,-0.0058,46.5,2
4,Ituri,Mongbalu,CD5405ZS10,150162,726,352,2026-05-18,1,483.48,2.685,...,0.6947,0.7629,-0.0682,0.7652,0.7964,-0.0311,0,-0.0682,55.1,65


In [3]:
features_A = ['km_to_mongbwalu', 'rwi_mean', 'travel_time_min', 'pop_density_per_km2']
features_B = features_A + ['stunting_2013_pct', 'ndvi_anomaly_filled']

X_A = df[features_A]
X_B = df[features_B]
y = df['log10_cases_per_100k_plus1']

In [5]:
from sklearn.model_selection import train_test_split

XA_train, XA_test, XB_train, XB_test, y_train, y_test = train_test_split(
    X_A, X_B, y, test_size=0.25, random_state=42)

In [6]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

model_A = LinearRegression().fit(XA_train, y_train)
model_B = LinearRegression().fit(XB_train, y_train)

print('Baseline:', mean_squared_error(y_test, [y_train.mean()] * len(y_test)))
print('Model A:', mean_squared_error(y_test, model_A.predict(XA_test)))
print('Model B:', mean_squared_error(y_test, model_B.predict(XB_test)))

Baseline: 0.5554886799969847
Model A: 0.3532132143527467
Model B: 0.34825309824712086


In [7]:
from sklearn.model_selection import RepeatedKFold, cross_val_score
import numpy as np

cv = RepeatedKFold(n_splits=5, n_repeats=20, random_state=42)

scores_A = -cross_val_score(LinearRegression(), X_A, y, cv=cv, scoring='neg_mean_squared_error')
scores_B = -cross_val_score(LinearRegression(), X_B, y, cv=cv, scoring='neg_mean_squared_error')

print('Model A average error:', scores_A.mean())
print('Model B average error:', scores_B.mean())
print('Model B won', (scores_B < scores_A).sum(), 'out of', len(scores_A), 'rounds')

Model A average error: 0.4670892913473507
Model B average error: 0.4609202151441242
Model B won 62 out of 100 rounds
